# 02 — AI Regime + Scenarios (workflow_update)

**NON_BASELINE_RUN**. Timed subprocesses; S=5000, N=30.

In [1]:
import subprocess
import time
from pathlib import Path

ROOT = Path.cwd().resolve()
while not (ROOT / "CLAUDE.md").exists():
    ROOT = ROOT.parent
BASE = [
    "--config",
    "configs/base.yaml",
    "--profile",
    "configs/workflow_update.yaml",
    "--override",
    "configs/workflow_update.yaml",
]


def run(name, cmd):
    print("$", " ".join(cmd), flush=True)
    t = time.perf_counter()
    p = subprocess.run(cmd, cwd=ROOT, check=False)
    elapsed = time.perf_counter() - t
    print(f"[{name}] exit={p.returncode} elapsed={elapsed:.2f}s")
    if p.returncode:
        raise RuntimeError(f"{name} failed")
    return elapsed

In [2]:
timings = {}
timings["regime"] = run("regime", ["uv", "run", "qshield-ai", "regime", *BASE])
timings["scenarios"] = run("scenarios", ["uv", "run", "qshield-ai", "scenarios", *BASE])
print(timings, "total", sum(timings.values()))

$ uv run qshield-ai regime --config configs/base.yaml --profile configs/workflow_update.yaml --override configs/workflow_update.yaml


Regime corr panel loại 1 ticker cold-start/không có train: ['VPL']. Scenarios vẫn dùng đủ 30 mã universe.
Feature frame: 1265 dòng, 2021-06-21 → 2026-07-30


Model is not converging.  Current: -1829.336432574885 is not greater than -1829.3363755365265. Delta is -5.703835859094397e-05
Model is not converging.  Current: -1725.9336489513362 is not greater than -1725.9335779973483. Delta is -7.095398791534535e-05


Champion seed=303, 1265 dòng regime đã ghi.


[regime] OK — 1265 dòng → artifacts/dev/regime


[regime] exit=0 elapsed=11.34s
$ uv run qshield-ai scenarios --config configs/base.yaml --profile configs/workflow_update.yaml --override configs/workflow_update.yaml


Ngày đánh giá t=2026-07-30, regime mục tiêu=volatile


Bỏ qua regime stress: pool rỗng, lý do loại: {'anchor_off_calendar': 0, 'beyond_evaluation_date': 0, 'incomplete_panel': 259}


[scenarios] PASS — cube (5000, 20, 30) regime=volatile t=2026-07-30 → artifacts/dev/scenarios


[scenarios] exit=0 elapsed=4.09s
{'regime': 11.33812187500007, 'scenarios': 4.086926249999578} total 15.425048124999648


In [3]:
import json

import numpy as np

p = ROOT / "artifacts/dev/scenarios"
m = json.loads((p / "scenario_manifest.json").read_text())
with np.load(p / "stress_scenarios.npz") as z:
    print("shape", z["scenarios"].shape)
print(
    "gate",
    m.get("gate_status"),
    "regime",
    m.get("target_regime"),
    "S",
    m.get("num_scenarios"),
)

shape (5000, 20, 30)
gate PASS regime volatile S 5000
